# PlantVillage — train on Colab (free GPU)

Trains the 15-class plant-disease model (two-phase MobileNetV2 + class weighting),
evaluates on a clean split, and downloads `plant_model.keras` + `class_names.json`
to drop into `backend/model/`.

**First:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.

Data comes from the Kaggle dataset `emmarex/plantdisease` — the exact 15 classes
(Pepper / Potato / Tomato) your local copy uses, so `class_names.json` will match
your backend unchanged.

In [ ]:
# 1) Check GPU + libraries (TF / sklearn / matplotlib are preinstalled on Colab)
!nvidia-smi -L
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "No GPU! Runtime > Change runtime type > T4 GPU"

## 2) Get the dataset (Kaggle)

Get `kaggle.json`: kaggle.com → your profile → **Settings** → **Create New API Token**
(downloads `kaggle.json`). Run the cell, then upload that file when prompted.

*(No Kaggle account? See the Google Drive alternative at the bottom.)*

In [ ]:
from google.colab import files
print("Upload kaggle.json ...")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d emmarex/plantdisease
!unzip -q -o plantdisease.zip -d data_raw
!ls data_raw

In [ ]:
# 3) Split into a CLEAN 80/20 train/val (wipes any prior split first -> no leakage)
import os, random, shutil

OUT, VAL_RATIO, SEED = "data", 0.2, 42
EXTS = {".jpg", ".jpeg", ".png"}
random.seed(SEED)

def is_image(n):
    return os.path.splitext(n)[1].lower() in EXTS

def find_src(root):
    """Find the folder whose subdirectories directly contain class images."""
    for dp, dirs, _ in os.walk(root):
        sub = [d for d in dirs if any(is_image(f) for f in os.listdir(os.path.join(dp, d)))]
        if len(sub) >= 10:
            return dp
    return root

SRC = find_src("data_raw")
print("Source:", SRC)

for split in ("train", "val"):
    p = os.path.join(OUT, split)
    if os.path.isdir(p):
        shutil.rmtree(p)

classes = []
for entry in sorted(os.listdir(SRC)):
    path = os.path.join(SRC, entry)
    if not os.path.isdir(path):
        continue
    imgs = [f for f in os.listdir(path) if is_image(f)]
    if imgs:
        classes.append((entry, imgs))

tr = va = 0
for name, imgs in classes:
    random.shuffle(imgs)
    nv = max(1, int(len(imgs) * VAL_RATIO))
    val_i, train_i = imgs[:nv], imgs[nv:]
    for split, lst in (("train", train_i), ("val", val_i)):
        d = os.path.join(OUT, split, name)
        os.makedirs(d, exist_ok=True)
        for im in lst:
            shutil.copy2(os.path.join(SRC, name, im), os.path.join(d, im))
    tr += len(train_i); va += len(val_i)
    print(f"  {name}: {len(train_i)} train / {len(val_i)} val")
print(f"\n{len(classes)} classes, {tr} train / {va} val")

In [ ]:
# 4) Train: two-phase MobileNetV2 with inverse-frequency class weights
import json
import tensorflow as tf
from tensorflow.keras import layers, models

IMG, BATCH, DATA = (224, 224), 32, "data"

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA + "/train", image_size=IMG, batch_size=BATCH, label_mode="categorical")
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA + "/val", image_size=IMG, batch_size=BATCH, label_mode="categorical")
class_names = train_ds.class_names
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
print(len(class_names), "classes:", class_names)

# inverse-frequency class weights (protect small classes like Potato_healthy)
counts = [sum(1 for x in os.listdir(os.path.join(DATA, "train", n))
              if x.lower().endswith(tuple(EXTS))) for n in class_names]
total, nc = sum(counts), len(class_names)
class_weight = {i: total / (nc * c) for i, c in enumerate(counts)}
for i, n in enumerate(class_names):
    print(f"  {n}: {counts[i]} -> {class_weight[i]:.2f}")

base = tf.keras.applications.MobileNetV2(
    input_shape=IMG + (3,), include_top=False, weights="imagenet")
base.trainable = False
aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"), layers.RandomRotation(0.1), layers.RandomZoom(0.1)])
inp = layers.Input(shape=IMG + (3,))
x = aug(inp)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
out = layers.Dense(nc, activation="softmax")(x)
model = models.Model(inp, out)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

print("\nPhase 1: frozen base (10 epochs)")
model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight)

print("\nPhase 2: fine-tune top 30 layers (5 epochs, lr=1e-5)")
base.trainable = True
for l in base.layers[:-30]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weight)

model.save("plant_model.keras")
with open("class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)
print("\nSaved plant_model.keras + class_names.json")

In [ ]:
# 5) Evaluate on the clean val split (this is the HONEST number)
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score

eval_ds = tf.keras.utils.image_dataset_from_directory(
    DATA + "/val", image_size=IMG, batch_size=BATCH, label_mode="categorical",
    shuffle=False, class_names=class_names)

y_true, y_pred = [], []
for imgs, labels in eval_ds:
    p = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(p, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

acc = accuracy_score(y_true, y_pred)
macro = f1_score(y_true, y_pred, average="macro")
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)  |  macro-F1: {macro:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))

In [ ]:
# 6) Download the model + class names -> put both into backend/model/ locally
from google.colab import files
files.download("plant_model.keras")
files.download("class_names.json")

---
### Alternative: no Kaggle account

Zip your local `ml/data/PlantVillage` folder, upload it to Google Drive, then run:
```python
from google.colab import drive
drive.mount("/content/drive")
!unzip -q -o "/content/drive/MyDrive/PlantVillage.zip" -d data_raw
```
Then skip cell 2 and continue from cell 3 (the splitter auto-detects the class folder).